In [1]:
import mlflow 
import pandas as pd 
import mlflow.sklearn 
from sklearn.feature_extraction.text import CountVectorizer 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score 
import re 
import string 
import nltk 
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 
import numpy as np 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)   #taking 500 samples for experimentation for faster training initially
df.to_csv('data.csv', index = False)  #uss sample ka csv file bana dega jahape ye notebook chal raha
df.head()

,review,sentiment
421,"Yep, lots of shouting, screaming, cheering, ar...",negative
505,"I'll be honest with yall, I was a junior in hi...",positive
321,I liked SOLINO very much. It is a very heart-r...,positive
475,"After seeing Forever Hollywood, it would be na...",negative
205,"The show had great episodes, this is not one o...",negative


In [3]:
#Define text preprocessing functions 
def lemmatization(text):
    """Lemmatize the input text using WordNet Lemmatizer""" 
    lemmatizer = WordNetLemmatizer()
    text = text.split() 
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text) 


def remove_stop_words(text):
    """Remove stop words from the input text"""
    stop_words = set(stopwords.words('english'))
    text = [word for word in str(text).split() if word not in stop_words] 
    return " ".join(text) 


def removing_numbers(text):
    """Remove numbers from the input text"""
    text = ''.join([char for char in text if not char.isdigit()])
    return text 


def lower_case(text):
    """Convert the input text to lowercase"""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)
#this method removes extra whitespaces/newlines as well between words if it exists


def remove_punctuation(text):
    """Remove punctuation from the text""" 
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text


def removing_urls(text):
    """Remove URLs from the text""" 
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)


def normalize_text(df):  #master function that runs all the function together by calling this func
    """Apply the functions to the dataframe"""
    try:
        df["review"] = df["review"].apply(lower_case)
        df['review'] = df['review'].apply(remove_punctuation)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(removing_numbers) 
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
421,yep lot shouting screaming cheering arguing ce...,negative
505,honest yall junior high school sitcom first ai...,positive
321,liked solino much heart rending story italian ...,positive
475,seeing forever hollywood would natural want se...,negative
205,show great episode one terrible episode hard f...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    259
positive    241
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive', 'negative'])
df = df[x]   #Keep only the rows where x is True and overwrite the old dataframe

We run this code kyuki value counts() func meh agar sentiment meh null values rahenge toh vo return nhi krega uska count sahi se

Hence, we run this block to ensure we are clear for what we want. 

Jo x rehta h, that is nothing but ek filtering boolean jo data ke saare rows ko dekta hai and check karta hai if sentiment k andhar ka value positive hai ya negative, and unhiko sirf retrieve karta hai.

If positive/negative hai toh `True` hai, else `False` 

In [7]:
df['sentiment'] = df['sentiment'].map({"positive": 1, "negative": 0})
df.head()

,review,sentiment
421,yep lot shouting screaming cheering arguing ce...,0
505,honest yall junior high school sitcom first ai...,1
321,liked solino much heart rending story italian ...,1
475,seeing forever hollywood would natural want se...,0
205,show great episode one terrible episode hard f...,0


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [13]:
#loading our vectorizer and splitting data into X and Y
vectorizer = CountVectorizer(max_features = 100) 
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

CountVectorizer is used to only grab the top 50 words that have been repeated the most throughout the vocab dictionary

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

In [15]:
import dagshub 

mlflow.set_tracking_uri("https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow")
dagshub.init(repo_owner="MLayush-dubey", repo_name="MLOps-IMDB-Sentiment-Analysis", mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

Initialized MLflow to track repo "MLayush-dubey/MLOps-IMDB-Sentiment-Analysis"

2026-04-13 18:25:54,720 - INFO - Initialized MLflow to track repo "MLayush-dubey/MLOps-IMDB-Sentiment-Analysis"


Repository MLayush-dubey/MLOps-IMDB-Sentiment-Analysis initialized!

2026-04-13 18:25:54,722 - INFO - Repository MLayush-dubey/MLOps-IMDB-Sentiment-Analysis initialized!


<Experiment: artifact_location='mlflow-artifacts:/d145f6962485479086c6f996580305a2', creation_time=1776085692608, experiment_id='0', last_update_time=1776085692608, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}>

Uppar MLFlow ka tracking url vagere setup krdiye before we fit our model

In [16]:
import mlflow 
import logging 
import os 
import time 
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score 

#configure logging 
logging.basicConfig(level=logging.INFO, format = "%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLFlow run...")

with mlflow.start_run():   #by running this; mlflow will start tracking onwards
    start_time = time.time() 

    try:
        logging.info("Loading preprocessing features")
        #preprocessing features ko log kr rahe hai MLflow m
        mlflow.log_param("vectorizer", "Bag Of Words")
        mlflow.log_param("num_features", 100)
        mlflow.log_param("test_size", 0.25)
        
        logging.info("Initializing Logistic Regression Model...")
        model = LogisticRegression(max_iter = 1000)   #increase max iter to prevent non convergence issues

        logging.info("Fitting the model")
        model.fit(X_train, y_train)

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")  #logging model params to MLFlow

        logging.info("Making predictions...")
        y_pred = model.predict(X_test) 

        logging.info("Calculating eval metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred) 
        precision = precision_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred) 

        logging.info("Logging evaluation metrics...")
        #eval metrics ko log kr rahe MLFlow meh
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)

        logging.info("Saving and logging the model")
        mlflow.sklearn.log_model(model, "model")

        #log execution time 
        end_time = time.time() 
        logging.info(f"Execution time: {end_time - start_time:.2f} seconds.")

        #print the results for verification 
        print(f"Accuracy: {accuracy}")
        print(f"Precision: {precision}")
        print(f"Recall: {recall}")
        print(f"F1-Score: {f1}")


    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info = True)  #exc_info gives full error detail including the line it occurred

2026-04-13 18:25:55,458 - INFO - Starting MLFlow run...
2026-04-13 18:25:55,938 - INFO - Loading preprocessing features
2026-04-13 18:25:56,764 - INFO - Initializing Logistic Regression Model...
2026-04-13 18:25:56,767 - INFO - Fitting the model
2026-04-13 18:25:56,777 - INFO - Logging model parameters...
2026-04-13 18:25:57,090 - INFO - Making predictions...
2026-04-13 18:25:57,091 - INFO - Calculating eval metrics...
2026-04-13 18:25:57,098 - INFO - Logging evaluation metrics...
2026-04-13 18:25:58,173 - INFO - Saving and logging the model
2026/04/13 18:26:00 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026-04-13 18:26:03,821 - INFO - Execution time: 7.88 seconds.


Accuracy: 0.616
Precision: 0.6065573770491803
Recall: 0.6065573770491803
F1-Score: 0.6065573770491803
🏃 View run kindly-moth-159 at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/0/runs/524dfaec8a604454a2ece8cabe264738
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/0
